## (Calculations are shown only for CHOL = 0.1 , 0.25 and 0.35. The user may use the code for other concentrations if desired)

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import numpy as np
import MDAnalysis as mda
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from code_libraries.counts_rdfs_4 import *
from code_libraries.predict_acyl_chains_NN_by_chain import *

In [ ]:
start_frame , stop_frame = 0 , 4000
sys_type = "sym"
sys_name_90 = "dopc_90"
sys_name_85 = "dopc_85"
sys_name_75 = "dopc_75"
sys_name_70 = "dopc_70"
sys_name_65 = "dopc_65"

In [ ]:
def predictions_cutoff_pbc_chol_reference(pbc_array, pred_array):

    label_1_all = []
    label_0_all = []

    for f in range(len(pbc_array)):

        pbc_frame = pbc_array[f]  # shape [no_chol, no_dopc]
        pred_frame = pred_array[f]  # shape [no_dopc]

        # Select columns (DOPCs) based on their classification
        label_1 = pbc_frame[:, pred_frame == True]   # all CHOLs, only Lo DOPCs
        label_0 = pbc_frame[:, pred_frame == False]  # all CHOLs, only Ld DOPCs

        label_1_all.append(label_1)
        label_0_all.append(label_0)

    return label_1_all, label_0_all

In [ ]:
data_array_dir = "../heavy_data_NO_GIT/data_arrays_by_chain_nn/"

## 5 pt

In [ ]:
dopc_90_box_dim = np.load(os.path.join(data_array_dir , "dopc_90_sn1_ft_last_5pt_box_dim.npy"))
dopc_90_5pt_sn1_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_90_sn1_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)
dopc_90_5pt_sn2_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_90_sn2_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)

dopc_85_box_dim = np.load(os.path.join(data_array_dir , "dopc_85_sn1_ft_last_5pt_box_dim.npy"))
dopc_85_5pt_sn1_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_85_sn1_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)
dopc_85_5pt_sn2_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_85_sn2_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)

dopc_75_box_dim = np.load(os.path.join(data_array_dir , "dopc_75_sn1_ft_last_5pt_box_dim.npy"))
dopc_75_5pt_sn1_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_75_sn1_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)
dopc_75_5pt_sn2_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_75_sn2_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)

dopc_70_box_dim = np.load(os.path.join(data_array_dir , "dopc_70_sn1_ft_last_5pt_box_dim.npy"))
dopc_70_5pt_sn1_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_70_sn1_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)
dopc_70_5pt_sn2_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_70_sn2_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)

dopc_65_box_dim = np.load(os.path.join(data_array_dir , "dopc_65_sn1_ft_last_5pt_box_dim.npy"))
dopc_65_5pt_sn1_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_65_sn1_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)
dopc_65_5pt_sn2_prediction_labels = np.load(os.path.join(data_array_dir , "dopc_65_sn2_ft_last_5pt_prediction_labels.npy") , allow_pickle=True)

In [ ]:
###load in system dimensions

sys_90_box_dim = np.load(os.path.join(data_array_dir , "dopc_90_sn1_ft_last_5pt_box_dim.npy"))
sys_90_box_dim = np.mean(sys_90_box_dim , axis = 0)

#load in raw, unaveraged dopc and chol positions; will have shape (2 , 2000 , ##lipids , ##atoms_per_lipid , 3)
##(2) corresponds to leaflet(0 is lower , 1 is upper)

dopc_90_pos_raw = clf_lo_ld_by_chain(sys_name_90 , sys_type , "raw" , "nn" , "sn1" , 3 , start_frame , stop_frame).bilayer_all_dopc_data()
chol_90_pos_raw = clf_lo_ld_by_chain(sys_name_90 , sys_type , "raw" , "nn" , "sn1", 3 , start_frame , stop_frame).bilayer_all_chol_data()

###coms's of cholesterol; no running average at this point

chol_90_lower_com_raw = chol_com_all(chol_90_pos_raw[0])
chol_90_upper_com_raw = chol_com_all(chol_90_pos_raw[1])


dopc_90_pos_raw_lower , dopc_90_pos_raw_upper  = dopc_90_pos_raw[0] , dopc_90_pos_raw[1]

###coms's of dopc by individual acyl chains; no running average at this point
dopc_90_sn1_pos_com_lower , dopc_90_sn1_pos_com_upper = acyl_chain_one_com(dopc_90_pos_raw_lower , "sn1"), acyl_chain_one_com(dopc_90_pos_raw_upper , "sn1")
dopc_90_sn2_pos_com_lower , dopc_90_sn2_pos_com_upper = acyl_chain_one_com(dopc_90_pos_raw_lower , "sn2"), acyl_chain_one_com(dopc_90_pos_raw_upper , "sn2")


###lists of pbc distances over all ferames; no running averages for now

pbc_distances_90_sn1_lower = compute_distances_PBC_chol_to_acyl_all_ts(chol_90_lower_com_raw , dopc_90_sn1_pos_com_lower , sys_90_box_dim)
pbc_distances_90_sn1_upper = compute_distances_PBC_chol_to_acyl_all_ts(chol_90_upper_com_raw , dopc_90_sn1_pos_com_upper , sys_90_box_dim)
pbc_distances_90_sn2_lower = compute_distances_PBC_chol_to_acyl_all_ts(chol_90_lower_com_raw , dopc_90_sn2_pos_com_lower , sys_90_box_dim)
pbc_distances_90_sn2_upper = compute_distances_PBC_chol_to_acyl_all_ts(chol_90_upper_com_raw , dopc_90_sn2_pos_com_upper , sys_90_box_dim)

####this is where running averages happen

pbc_distances_5pt_90_sn1_lower = convolve_pbc_list(pbc_distances_90_sn1_lower , 5)
pbc_distances_5pt_90_sn1_upper = convolve_pbc_list(pbc_distances_90_sn1_upper , 5)
pbc_distances_5pt_90_sn2_lower = convolve_pbc_list(pbc_distances_90_sn2_lower , 5)
pbc_distances_5pt_90_sn2_upper = convolve_pbc_list(pbc_distances_90_sn2_upper , 5)

# ##now filter lo/ld out by chains as intended

pbc_ordered_5pt_90_sn1_lower , pbc_disordered_5pt_90_sn1_lower = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_90_sn1_lower , dopc_90_5pt_sn1_prediction_labels[0])
pbc_ordered_5pt_90_sn1_upper , pbc_disordered_5pt_90_sn1_upper = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_90_sn1_upper , dopc_90_5pt_sn1_prediction_labels[1])
pbc_ordered_5pt_90_sn2_lower , pbc_disordered_5pt_90_sn2_lower = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_90_sn2_lower , dopc_90_5pt_sn2_prediction_labels[0])
pbc_ordered_5pt_90_sn2_upper , pbc_disordered_5pt_90_sn2_upper = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_90_sn2_upper , dopc_90_5pt_sn2_prediction_labels[1])

###rdf as usual

dr = 0.05
radius = 2

radial_distances_90 = np.arange(0 , radius , dr)

disordered_rdf_sn1_lower_90 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_90_sn1_lower , dr , radius ,sys_90_box_dim )
disordered_rdf_sn1_upper_90 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_90_sn1_upper , dr , radius , sys_90_box_dim)

ordered_rdf_sn1_lower_90 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_90_sn1_lower , dr , radius , sys_90_box_dim)
ordered_rdf_sn1_upper_90 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_90_sn1_upper , dr , radius , sys_90_box_dim)

av_90_sn1_0_rdf = 0.5*(np.array(disordered_rdf_sn1_upper_90) + np.array(disordered_rdf_sn1_lower_90))
av_90_sn1_1_rdf = 0.5*(np.array(ordered_rdf_sn1_upper_90) + np.array(ordered_rdf_sn1_lower_90))

disordered_rdf_sn2_lower_90 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_90_sn2_lower , dr , radius ,sys_90_box_dim )
disordered_rdf_sn2_upper_90 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_90_sn2_upper , dr , radius , sys_90_box_dim)

ordered_rdf_sn2_lower_90 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_90_sn2_lower , dr , radius , sys_90_box_dim)
ordered_rdf_sn2_upper_90 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_90_sn2_upper , dr , radius , sys_90_box_dim)

av_90_sn2_0_rdf = 0.5*(np.array(disordered_rdf_sn2_upper_90) + np.array(disordered_rdf_sn2_lower_90))
av_90_sn2_1_rdf = 0.5*(np.array(ordered_rdf_sn2_upper_90) + np.array(ordered_rdf_sn2_lower_90))

In [ ]:
###load in system dimensions

sys_75_box_dim = np.load(os.path.join(data_array_dir , "dopc_75_sn1_ft_last_5pt_box_dim.npy"))
sys_75_box_dim = np.mean(sys_75_box_dim , axis = 0)

#load in raw, unaveraged dopc and chol positions; will have shape (2 , 2000 , ##lipids , ##atoms_per_lipid , 3)
##(2) corresponds to leaflet(0 is lower , 1 is upper)

dopc_75_pos_raw = clf_lo_ld_by_chain(sys_name_75 , sys_type , "raw" , "nn" , "sn1" , 3 , start_frame , stop_frame).bilayer_all_dopc_data()
chol_75_pos_raw = clf_lo_ld_by_chain(sys_name_75 , sys_type , "raw" , "nn" , "sn1", 3 , start_frame , stop_frame).bilayer_all_chol_data()

###coms's of cholesterol; no running average at this point

chol_75_lower_com_raw = chol_com_all(chol_75_pos_raw[0])
chol_75_upper_com_raw = chol_com_all(chol_75_pos_raw[1])


dopc_75_pos_raw_lower , dopc_75_pos_raw_upper  = dopc_75_pos_raw[0] , dopc_75_pos_raw[1]

###coms's of dopc by individual acyl chains; no running average at this point
dopc_75_sn1_pos_com_lower , dopc_75_sn1_pos_com_upper = acyl_chain_one_com(dopc_75_pos_raw_lower , "sn1"), acyl_chain_one_com(dopc_75_pos_raw_upper , "sn1")
dopc_75_sn2_pos_com_lower , dopc_75_sn2_pos_com_upper = acyl_chain_one_com(dopc_75_pos_raw_lower , "sn2"), acyl_chain_one_com(dopc_75_pos_raw_upper , "sn2")


###lists of pbc distances over all ferames; no running averages for now

pbc_distances_75_sn1_lower = compute_distances_PBC_chol_to_acyl_all_ts(chol_75_lower_com_raw , dopc_75_sn1_pos_com_lower , sys_75_box_dim)
pbc_distances_75_sn1_upper = compute_distances_PBC_chol_to_acyl_all_ts(chol_75_upper_com_raw , dopc_75_sn1_pos_com_upper , sys_75_box_dim)
pbc_distances_75_sn2_lower = compute_distances_PBC_chol_to_acyl_all_ts(chol_75_lower_com_raw , dopc_75_sn2_pos_com_lower , sys_75_box_dim)
pbc_distances_75_sn2_upper = compute_distances_PBC_chol_to_acyl_all_ts(chol_75_upper_com_raw , dopc_75_sn2_pos_com_upper , sys_75_box_dim)

####this is where running averages happen

pbc_distances_5pt_75_sn1_lower = convolve_pbc_list(pbc_distances_75_sn1_lower , 5)
pbc_distances_5pt_75_sn1_upper = convolve_pbc_list(pbc_distances_75_sn1_upper , 5)
pbc_distances_5pt_75_sn2_lower = convolve_pbc_list(pbc_distances_75_sn2_lower , 5)
pbc_distances_5pt_75_sn2_upper = convolve_pbc_list(pbc_distances_75_sn2_upper , 5)

# ##now filter lo/ld out by chains as intended

pbc_ordered_5pt_75_sn1_lower , pbc_disordered_5pt_75_sn1_lower = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_75_sn1_lower , dopc_75_5pt_sn1_prediction_labels[0])
pbc_ordered_5pt_75_sn1_upper , pbc_disordered_5pt_75_sn1_upper = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_75_sn1_upper , dopc_75_5pt_sn1_prediction_labels[1])
pbc_ordered_5pt_75_sn2_lower , pbc_disordered_5pt_75_sn2_lower = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_75_sn2_lower , dopc_75_5pt_sn2_prediction_labels[0])
pbc_ordered_5pt_75_sn2_upper , pbc_disordered_5pt_75_sn2_upper = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_75_sn2_upper , dopc_75_5pt_sn2_prediction_labels[1])

###rdf as usual

dr = 0.05
radius = 2

radial_distances_75 = np.arange(0 , radius , dr)

disordered_rdf_sn1_lower_75 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_75_sn1_lower , dr , radius ,sys_75_box_dim )
disordered_rdf_sn1_upper_75 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_75_sn1_upper , dr , radius , sys_75_box_dim)

ordered_rdf_sn1_lower_75 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_75_sn1_lower , dr , radius , sys_75_box_dim)
ordered_rdf_sn1_upper_75 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_75_sn1_upper , dr , radius , sys_75_box_dim)

av_75_sn1_0_rdf = 0.5*(np.array(disordered_rdf_sn1_upper_75) + np.array(disordered_rdf_sn1_lower_75))
av_75_sn1_1_rdf = 0.5*(np.array(ordered_rdf_sn1_upper_75) + np.array(ordered_rdf_sn1_lower_75))

disordered_rdf_sn2_lower_75 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_75_sn2_lower , dr , radius ,sys_75_box_dim )
disordered_rdf_sn2_upper_75 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_75_sn2_upper , dr , radius , sys_75_box_dim)

ordered_rdf_sn2_lower_75 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_75_sn2_lower , dr , radius , sys_75_box_dim)
ordered_rdf_sn2_upper_75 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_75_sn2_upper , dr , radius , sys_75_box_dim)

av_75_sn2_0_rdf = 0.5*(np.array(disordered_rdf_sn2_upper_75) + np.array(disordered_rdf_sn2_lower_75))
av_75_sn2_1_rdf = 0.5*(np.array(ordered_rdf_sn2_upper_75) + np.array(ordered_rdf_sn2_lower_75))

In [ ]:
###load in system dimensions

sys_65_box_dim = np.load(os.path.join(data_array_dir , "dopc_65_sn1_ft_last_5pt_box_dim.npy"))
sys_65_box_dim = np.mean(sys_65_box_dim , axis = 0)

#load in raw, unaveraged dopc and chol positions; will have shape (2 , 2000 , ##lipids , ##atoms_per_lipid , 3)
##(2) corresponds to leaflet(0 is lower , 1 is upper)

dopc_65_pos_raw = clf_lo_ld_by_chain(sys_name_65 , sys_type , "raw" , "nn" , "sn1" , 3 , start_frame , stop_frame).bilayer_all_dopc_data()
chol_65_pos_raw = clf_lo_ld_by_chain(sys_name_65 , sys_type , "raw" , "nn" , "sn1", 3 , start_frame , stop_frame).bilayer_all_chol_data()

###coms's of cholesterol; no running average at this point

chol_65_lower_com_raw = chol_com_all(chol_65_pos_raw[0])
chol_65_upper_com_raw = chol_com_all(chol_65_pos_raw[1])


dopc_65_pos_raw_lower , dopc_65_pos_raw_upper  = dopc_65_pos_raw[0] , dopc_65_pos_raw[1]

###coms's of dopc by individual acyl chains; no running average at this point
dopc_65_sn1_pos_com_lower , dopc_65_sn1_pos_com_upper = acyl_chain_one_com(dopc_65_pos_raw_lower , "sn1"), acyl_chain_one_com(dopc_65_pos_raw_upper , "sn1")
dopc_65_sn2_pos_com_lower , dopc_65_sn2_pos_com_upper = acyl_chain_one_com(dopc_65_pos_raw_lower , "sn2"), acyl_chain_one_com(dopc_65_pos_raw_upper , "sn2")


###lists of pbc distances over all ferames; no running averages for now

pbc_distances_65_sn1_lower = compute_distances_PBC_chol_to_acyl_all_ts(chol_65_lower_com_raw , dopc_65_sn1_pos_com_lower , sys_65_box_dim)
pbc_distances_65_sn1_upper = compute_distances_PBC_chol_to_acyl_all_ts(chol_65_upper_com_raw , dopc_65_sn1_pos_com_upper , sys_65_box_dim)
pbc_distances_65_sn2_lower = compute_distances_PBC_chol_to_acyl_all_ts(chol_65_lower_com_raw , dopc_65_sn2_pos_com_lower , sys_65_box_dim)
pbc_distances_65_sn2_upper = compute_distances_PBC_chol_to_acyl_all_ts(chol_65_upper_com_raw , dopc_65_sn2_pos_com_upper , sys_65_box_dim)

####this is where running averages happen

pbc_distances_5pt_65_sn1_lower = convolve_pbc_list(pbc_distances_65_sn1_lower , 5)
pbc_distances_5pt_65_sn1_upper = convolve_pbc_list(pbc_distances_65_sn1_upper , 5)
pbc_distances_5pt_65_sn2_lower = convolve_pbc_list(pbc_distances_65_sn2_lower , 5)
pbc_distances_5pt_65_sn2_upper = convolve_pbc_list(pbc_distances_65_sn2_upper , 5)

# ##now filter lo/ld out by chains as intended

pbc_ordered_5pt_65_sn1_lower , pbc_disordered_5pt_65_sn1_lower = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_65_sn1_lower , dopc_65_5pt_sn1_prediction_labels[0])
pbc_ordered_5pt_65_sn1_upper , pbc_disordered_5pt_65_sn1_upper = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_65_sn1_upper , dopc_65_5pt_sn1_prediction_labels[1])
pbc_ordered_5pt_65_sn2_lower , pbc_disordered_5pt_65_sn2_lower = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_65_sn2_lower , dopc_65_5pt_sn2_prediction_labels[0])
pbc_ordered_5pt_65_sn2_upper , pbc_disordered_5pt_65_sn2_upper = predictions_cutoff_pbc_chol_reference(pbc_distances_5pt_65_sn2_upper , dopc_65_5pt_sn2_prediction_labels[1])

###rdf as usual

dr = 0.05
radius = 2

radial_distances_65 = np.arange(0 , radius , dr)

disordered_rdf_sn1_lower_65 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_65_sn1_lower , dr , radius ,sys_65_box_dim )
disordered_rdf_sn1_upper_65 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_65_sn1_upper , dr , radius , sys_65_box_dim)

ordered_rdf_sn1_lower_65 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_65_sn1_lower , dr , radius , sys_65_box_dim)
ordered_rdf_sn1_upper_65 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_65_sn1_upper , dr , radius , sys_65_box_dim)

av_65_sn1_0_rdf = 0.5*(np.array(disordered_rdf_sn1_upper_65) + np.array(disordered_rdf_sn1_lower_65))
av_65_sn1_1_rdf = 0.5*(np.array(ordered_rdf_sn1_upper_65) + np.array(ordered_rdf_sn1_lower_65))

disordered_rdf_sn2_lower_65 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_65_sn2_lower , dr , radius ,sys_65_box_dim )
disordered_rdf_sn2_upper_65 = RDF_over_all_frames_diff_species(pbc_disordered_5pt_65_sn2_upper , dr , radius , sys_65_box_dim)

ordered_rdf_sn2_lower_65 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_65_sn2_lower , dr , radius , sys_65_box_dim)
ordered_rdf_sn2_upper_65 = RDF_over_all_frames_diff_species(pbc_ordered_5pt_65_sn2_upper , dr , radius , sys_65_box_dim)

av_65_sn2_0_rdf = 0.5*(np.array(disordered_rdf_sn2_upper_65) + np.array(disordered_rdf_sn2_lower_65))
av_65_sn2_1_rdf = 0.5*(np.array(ordered_rdf_sn2_upper_65) + np.array(ordered_rdf_sn2_lower_65))

## PLOTS HERE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24 , 5), sharey=True)
systems = [
    (radial_distances_90, av_90_sn1_0_rdf, av_90_sn1_1_rdf, av_90_sn2_0_rdf, av_90_sn2_1_rdf, r'$\mathbf{\chi_C = 0.1}$'),
    (radial_distances_75, av_75_sn1_0_rdf, av_75_sn1_1_rdf, av_75_sn2_0_rdf, av_75_sn2_1_rdf, r'$\mathbf{\chi_C = 0.25}$'),
    (radial_distances_65, av_65_sn1_0_rdf, av_65_sn1_1_rdf, av_65_sn2_0_rdf, av_65_sn2_1_rdf, r'$\mathbf{\chi_C = 0.35}$'),
]
for ax, (r, sn1_ld, sn1_lo, sn2_ld, sn2_lo, title) in zip(axes, systems):
    ax.plot(r, sn1_ld, color='blue', linewidth=2, label='sn-1 Ld')
    ax.plot(r, sn1_lo, color='green', linewidth=2, label='sn-1 Lo')
    ax.plot(r, sn2_ld, color='red', linewidth=2, label='sn-2 Ld')
    ax.plot(r, sn2_lo, color='black', linewidth=2, label='sn-2 Lo')
    #ax.hlines(1, 0, 2.5, linestyles='dotted', colors='gray', linewidth=2)
    ax.set_xlabel(r'$\mathbf{r \, (nm)}$', fontsize=25, fontweight='bold')
    ax.text(0.95, 0.05, title, transform=ax.transAxes, fontsize=25, fontweight='bold',
            verticalalignment='bottom', horizontalalignment='right')
    ax.set_xlim(0, 2.0)
    ax.set_ylim(0, 2.5)
    ax.set_xticks([0.5, 1.0, 1.5, 2.0])
    ax.tick_params(axis='both', which='major', labelsize=25, width=2.5, length=8, direction="inout")
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    legend = ax.legend(fontsize=18, prop={'weight':'bold', 'size': 22}, frameon=True, fancybox=True)
    for line in legend.get_lines():
        line.set_linewidth(3.0)
axes[0].set_ylabel(r'$\mathbf{g(r)}$', fontsize=25, fontweight='bold')
plt.tight_layout()
plt.show()